# Extração e auditoria do dataset de modelagem
Extrai a view privada do Supabase, verifica o contrato temporal e produz um snapshot reproduzível. Nenhuma credencial é escrita no notebook.

In [ ]:
%pip -q install pandas pyarrow sqlalchemy 'psycopg[binary]'


In [ ]:
import hashlib, os, sys
from datetime import datetime, timezone
from pathlib import Path
import pandas as pd
from sqlalchemy import create_engine, text

try:
    from google.colab import userdata
    DB_URL = userdata.get('TCC_DB_URL')
except Exception:
    DB_URL = os.environ.get('TCC_DB_URL')
if not DB_URL:
    raise RuntimeError('Cadastre TCC_DB_URL nos Segredos do Colab ou no ambiente.')

repo_candidates = [Path.cwd(), Path('/content/tcc-vigilancia-wbe-iot')]
repo_root = next((p for p in repo_candidates if (p / 'src').exists()), None)
if repo_root is None:
    raise RuntimeError('Clone/abra o repositório antes de executar o notebook.')
sys.path.insert(0, str(repo_root))
from src.validation import validate_dataset


In [ ]:
query = text('SELECT * FROM tcc_private.dataset_modelagem ORDER BY horizonte, inicio_semana_t')
engine = create_engine(DB_URL, pool_pre_ping=True, connect_args={'sslmode': 'require'})
with engine.connect() as connection:
    df = pd.read_sql(query, connection)
validate_dataset(df)
print('Contrato válido:', df.shape)
display(df.groupby(['horizonte', 'split'], observed=True).size().rename('linhas'))


In [ ]:
# Missingness por split: diferenças grandes devem ser investigadas antes do treino.
missing = (df.groupby('split', observed=True).apply(lambda x: x.isna().mean(), include_groups=False).T)
display(missing.sort_values(list(missing.columns), ascending=False).head(25))

# Drift descritivo sem tocar no alvo para seleção.
numeric = df.select_dtypes(include='number').columns.difference(['target_casos'])
summary = df.groupby('split', observed=True)[numeric].mean().T
if {'treino', 'validacao'}.issubset(summary.columns):
    train_std = df.loc[df.split.eq('treino'), numeric].std().replace(0, pd.NA)
    summary['drift_validacao_em_sd'] = ((summary.validacao - summary.treino) / train_std).abs()
display(summary.sort_values('drift_validacao_em_sd', ascending=False).head(20))


In [ ]:
# Grave fora do Git, por exemplo em uma pasta montada do Drive.
output_dir = Path('/content/drive/MyDrive/TCC/datasets')
output_dir.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
version = str(df['versao_dataset'].iloc[0])
path = output_dir / f'{version}_{stamp}.parquet'
df.to_parquet(path, index=False)
digest = hashlib.sha256(path.read_bytes()).hexdigest()
print('Snapshot:', path)
print('SHA-256:', digest)
